In [1]:
#proto mechanixm for the random seed
import random 
first_two_pixel_value=10
random.seed(first_two_pixel_value)
num_of_pixels=10 
rando_list = [i for i in range(num_of_pixels)]
random.Random(first_two_pixel_value).shuffle(rando_list)
rando_list

[5, 2, 7, 1, 8, 4, 3, 6, 0, 9]

In [2]:
#file imports used
from PIL import Image
from matplotlib import pyplot as plt
import math
import codecs

In [3]:
number_of_bits = 1
#open a bmp and get its size
x = Image.open("stego.bmp")
pic_size = x.size
num_of_pixels = x.size[0]*x.size[1]
print(num_of_pixels)
available_bytes=(num_of_pixels*(number_of_bits*3))/8
print(available_bytes,"bytes")

1091840
409440.0 bytes


In [4]:
#value of the first two pixels 
firstpixel = x.getpixel((0,0))
secondpixel = x.getpixel((0,1))

In [5]:
# proto mechanism for generating a seed from first two pixels
seed = math.floor((sum(firstpixel)/3+sum(secondpixel)/3)/2)

In [6]:
#Text file to binary and back
fileName="test_message.txt"
with open(fileName, mode='r') as file:
    fileContent = file.read()
print(fileContent)
res = ''.join(format(ord(i), '08b') for i in fileContent)
print(res)
ans = []
for i in range(0, len(res), 8):
    ans.append(chr(int(res[i:i+8],2)))
output = ''.join(ans)
output

This is a test message.  Hopefully this can get hidden in a file and come out intact on the other side.
01010100011010000110100101110011001000000110100101110011001000000110000100100000011101000110010101110011011101000010000001101101011001010111001101110011011000010110011101100101001011100010000000100000010010000110111101110000011001010110011001110101011011000110110001111001001000000111010001101000011010010111001100100000011000110110000101101110001000000110011101100101011101000010000001101000011010010110010001100100011001010110111000100000011010010110111000100000011000010010000001100110011010010110110001100101001000000110000101101110011001000010000001100011011011110110110101100101001000000110111101110101011101000010000001101001011011100111010001100001011000110111010000100000011011110110111000100000011101000110100001100101001000000110111101110100011010000110010101110010001000000111001101101001011001000110010100101110


'This is a test message.  Hopefully this can get hidden in a file and come out intact on the other side.'

In [7]:

class BitFeederString:
    """This class will take a file_name and convert it into string-based bits
    that can be extracted in small pieces.
    
    example syntax:
    bfeed = BitFeederString(my_message_file)
    first_three_bits = bfeed.get_bits(3)
    next_three_bits = bfeed.get_bits(3)
    following_two bits = bfeed.get_bits(2)
    """

    def __init__(self, file_name):       
        self.file_name = file_name
        self._load_bytes()
        self._make_bitstring()
        self.index = 0
        self.finished = False

    def _load_bytes(self):
        with open(self.file_name, 'rb') as in_file:
            self.bytes = in_file.read()

    def _make_bitstring(self):
        bitsarray = []
        for byte in self.bytes:
            bits = bin(byte)[2:]
            bits = '0'*(8-len(bits)) + bits
            bitsarray.append(bits)
        self.bitstring = ''.join(bitsarray)
    
    def get_length(self):
        return(len(self.bitstring))

    def get_bits(self, n, returntype=str):
        """this function returns n bits 
            
            x = bitfeederobj.get_bits(6, returntype=str)
            print(x)

            '101110'

            x = bitfeederobj.get_bits(6, returntype=int)
            print(x)

            46

        Args:
            n (int): Number of bits requested back
            returntype (type): The requested data return type. 
                Currently supports str and intDefaults to str.
                


        Returns:
            str: bit data
            or 
            int: bit data
            (return type depends on input parameter)
        """        
        start = self.index
        self.index += n
        if self.index < len(self.bitstring):
            r = self.bitstring[start:self.index]

        else:
            r = self.bitstring[start:]
            self.finished=True

        if returntype == int:
            return int(r, 2)
        return r

In [8]:
bfeed = BitFeederString("test_message.txt")
length = bfeed.get_length()
length

824

In [9]:
#Makes sure we have an 8 bit byte from int
def byte_maker(color):
    return f'{color:08b}'
#returns an int from byte bin string
def int_maker(color):
    return int(color, 2)
#encodes however many bits specified
def encoder(cover,message,bits):
    return cover[:-bits]+message

#PIXELS ENCODED
for i in range(pic_size[0]):
    for j in range(pic_size[1]):
        color = x.getpixel((i,j))
        modified = []
        print("OG"+str(color))
        for RGB in range(len(color)):
                input = color[RGB]
                #print(str(input))
                cover = byte_maker(input)
                #print(cover)
                message = bfeed.get_bits(1)
                #print(message)
                encoded = encoder(cover,message,1)
                #print(encoded)
                output = int(encoded, 2)
                #print(output)
                modified.append(output)
        x.putpixel( (i , j), (modified[0], modified[1], modified[2], 255))
        print("MOD"+str(x.getpixel((i,j))))
        if bfeed.finished == True:
            break

OG(178, 132, 145)
MOD(178, 133, 144)
OG(170, 124, 137)
MOD(171, 124, 137)
OG(182, 134, 148)
MOD(182, 134, 148)
OG(180, 132, 146)
MOD(181, 133, 146)
OG(181, 133, 147)
MOD(181, 132, 146)
OG(186, 138, 152)
MOD(186, 138, 153)
OG(175, 127, 143)
MOD(175, 126, 143)
OG(184, 136, 152)
MOD(184, 136, 153)
OG(185, 134, 151)
MOD(184, 135, 151)
OG(183, 132, 149)
MOD(183, 132, 148)
OG(179, 128, 145)
MOD(179, 129, 144)
OG(178, 127, 144)
MOD(178, 127, 144)
OG(172, 121, 138)
MOD(172, 120, 138)
OG(178, 127, 144)
MOD(178, 126, 145)
OG(181, 130, 149)
MOD(181, 130, 149)
OG(182, 131, 150)
MOD(182, 130, 151)
OG(181, 133, 149)
MOD(180, 133, 149)
OG(176, 129, 145)
MOD(177, 128, 144)
OG(180, 133, 149)
MOD(181, 133, 148)
OG(176, 129, 145)
MOD(176, 129, 144)
OG(177, 130, 146)
MOD(176, 130, 146)
OG(180, 133, 149)
MOD(180, 132, 149)
OG(175, 128, 144)
MOD(175, 128, 144)
OG(178, 131, 147)
MOD(178, 130, 147)
OG(174, 126, 142)
MOD(174, 126, 143)
OG(181, 133, 149)
MOD(180, 132, 148)
OG(185, 137, 153)
MOD(184, 136, 152)
O

MOD(34, 24, 35)
OG(74, 55, 77)
MOD(37, 27, 38)
OG(79, 60, 82)
MOD(39, 30, 41)
OG(72, 53, 75)
MOD(36, 26, 37)
OG(67, 48, 70)
MOD(33, 24, 35)
OG(71, 52, 74)
MOD(35, 26, 37)
OG(73, 54, 76)
MOD(36, 27, 38)
OG(69, 51, 73)
MOD(34, 25, 36)
OG(70, 52, 74)
MOD(35, 26, 37)
OG(71, 53, 75)
MOD(35, 26, 37)
OG(71, 53, 75)
MOD(35, 26, 37)
OG(69, 51, 73)
MOD(34, 25, 36)
OG(69, 51, 73)
MOD(34, 25, 36)
OG(71, 53, 75)
MOD(35, 26, 37)
OG(73, 55, 77)
MOD(36, 27, 38)
OG(69, 51, 73)
MOD(34, 25, 36)
OG(69, 51, 73)
MOD(34, 25, 36)
OG(78, 60, 82)
MOD(39, 30, 41)
OG(70, 52, 74)
MOD(35, 26, 37)
OG(75, 57, 79)
MOD(37, 28, 39)
OG(66, 48, 70)
MOD(33, 24, 35)
OG(72, 54, 76)
MOD(36, 27, 38)
OG(71, 54, 73)
MOD(35, 27, 36)
OG(76, 59, 78)
MOD(38, 29, 39)
OG(74, 57, 75)
MOD(37, 28, 37)
OG(72, 55, 73)
MOD(36, 27, 36)
OG(70, 53, 71)
MOD(35, 26, 35)
OG(70, 53, 71)
MOD(35, 26, 35)
OG(71, 54, 72)
MOD(35, 27, 36)
OG(70, 53, 72)
MOD(35, 26, 36)
OG(70, 53, 72)
MOD(35, 26, 36)
OG(70, 53, 72)
MOD(35, 26, 36)
OG(72, 55, 74)
MOD(36, 

MOD(50, 37, 55)
OG(96, 72, 108)
MOD(48, 36, 54)
OG(96, 72, 108)
MOD(48, 36, 54)
OG(99, 75, 111)
MOD(49, 37, 55)
OG(101, 77, 113)
MOD(50, 38, 56)
OG(116, 92, 128)
MOD(58, 46, 64)
OG(138, 114, 150)
MOD(69, 57, 75)
OG(110, 88, 124)
MOD(55, 44, 62)
OG(93, 71, 107)
MOD(46, 35, 53)
OG(87, 67, 102)
MOD(43, 33, 51)
OG(98, 78, 113)
MOD(49, 39, 56)
OG(100, 80, 115)
MOD(50, 40, 57)
OG(95, 75, 110)
MOD(47, 37, 55)
OG(94, 74, 109)
MOD(47, 37, 54)
OG(97, 75, 111)
MOD(48, 37, 55)
OG(104, 80, 116)
MOD(52, 40, 58)
OG(100, 74, 111)
MOD(50, 37, 55)
OG(99, 73, 110)
MOD(49, 36, 55)
OG(101, 75, 112)
MOD(50, 37, 56)
OG(103, 76, 111)
MOD(51, 38, 55)
OG(100, 73, 108)
MOD(50, 36, 54)
OG(100, 73, 106)
MOD(50, 36, 53)
OG(103, 76, 109)
MOD(51, 38, 54)
OG(98, 71, 106)
MOD(49, 35, 53)
OG(101, 74, 109)
MOD(50, 37, 54)
OG(102, 76, 111)
MOD(51, 38, 55)
OG(100, 74, 109)
MOD(50, 37, 54)
OG(100, 75, 114)
MOD(50, 37, 57)
OG(103, 78, 117)
MOD(51, 39, 58)
OG(101, 79, 117)
MOD(50, 39, 58)
OG(100, 78, 114)
MOD(50, 39, 57)
OG(1

In [10]:
#x

In [11]:
class BitSaver:
    """Saves bits extracted from a file. Can check to see if length of data read is expected amount of data
    bitsaver = BitSaver(file_length=1024) # bits

    bitsaver.add('1010001')
    ...
    bitsaver.add('111')

    bitsaver.bits = '1010001............111'

    bitsaver.save_file(file_name) # save bits to file

    while not bitsaver.finished:
        new_bits = read_data()
        bitsaver.add(new_bits)
    """    
    def __init__(self, file_length=None):
        self.bits = []
        self.file_length = file_length
        self.read_length = 0
        self.finished = False

    def add_bits(self, bits, datatype=str):
        if datatype == int:
            bits = bin(bits)[2:]
        
        self.bits.append(bits)
        self.read_length += len(bits)

        if self.file_length is not None:
            if self.read_length >= self.file_length:
                self.bits = ''.join(self.bits)
                self.bits = self.bits[:self.file_length]
                self.finished = True

    def save_file(self, file_name):
        self.bytes = bytearray()
        index = 0
        bit_len = len(self.bits)
        while index <= bit_len-7:
            self.bytes.append(int(self.bits[index:index+8],2))
            index += 8
        with open(file_name, 'wb') as out_file:
            out_file.write(self.bytes)

In [12]:
bs = BitSaver(file_length=len(bfeed.bytes)*8)

In [13]:
#BITSAVER DOES NOT LIKE ME ADDING 1 BIT AT A TIME
finaloutput = []
index = 0
for i in range(pic_size[0]):
    for j in range(pic_size[1]): 
        color = x.getpixel((i,j))
        for RGB in range(len(color)):
            output=byte_maker(color[RGB])
            print(output)
            bit = output[7:]
            print(bit)
            bs.add_bits(bit)
            index += 1
            if index > bfeed.get_length():
                break

10110010
0
10000101
1
10010000
0
10101011
1
01111100
0
10001001
1
10110110
0
10000110
0
10010100
0
10110101
1
10000101
1
10010010
0
10110101
1
10000100
0
10010010
0
10111010
0
10001010
0
10011001
1
10101111
1
01111110
0
10001111
1
10111000
0
10001000
0
10011001
1
10111000
0
10000111
1
10010111
1
10110111
1
10000100
0
10010100
0
10110011
1
10000001
1
10010000
0
10110010
0
01111111
1
10010000
0
10101100
0
01111000
0
10001010
0
10110010
0
01111110
0
10010001
1
10110101
1
10000010
0
10010101
1
10110110
0
10000010
0
10010111
1
10110100
0
10000101
1
10010101
1
10110001
1
10000000
0
10010000
0
10110101
1
10000101
1
10010100
0
10110000
0
10000001
1
10010000
0
10110000
0
10000010
0
10010010
0
10110100
0
10000100
0
10010101
1
10101111
1
10000000
0
10010000
0
10110010
0
10000010
0
10010011
1
10101110
0
01111110
0
10001111
1
10110100
0
10000100
0
10010100
0
10111000
0
10001000
0
10011000
0
10110101
1
10000101
1
10010101
1
10110010
0
10000011
1
10010010
0
10110100
0
10000100
0
10010101
1
10110101
1

AttributeError: 'str' object has no attribute 'append'

In [14]:
bs.save_file("ajdkla.txt")